# FormCoach - Analyser B training (1D-CNN / BiLSTM)

Trains a small deep-learning model that reads **one squat rep as a time series** (knee angle, hip angle, torso lean,
knee-over-toe, hip height vs knee) and predicts whether the judge would call it a **valid or invalid rep**.
Everything is tested **leave-one-person-out**: every rep is predicted by a model that never saw that person.
The rule-based depth thresholds are the baseline to beat.

**Data:** `datasets/sequences_cfrep.npz`, built from the CFRep dataset (per-rep labels by a certified CrossFit judge).
It is already in the repo, so you do **not** need the videos.

**Where to run it** (a GPU is optional - the models have only a few thousand parameters):

| Platform | Steps |
|---|---|
| **Google Colab** | Upload or open this notebook -> *Runtime > Run all*. (GPU optional: *Runtime > Change runtime type*.) |
| **Kaggle** | New notebook -> *File > Import notebook*. In the right-hand *Session options* turn **Internet ON** (needed to clone the repo). GPU optional. |
| **Local** | `pip install -r requirements.txt` then open it in Jupyter / VS Code from the repo root. |

If the repo is private, put a token in the URL: `https://<TOKEN>@github.com/Gulshan-heap/form-analyser.git`.

In [ ]:
# --- 1. Setup: get the repo code (works on Colab, Kaggle or locally) ---
import glob, os, shutil, subprocess, sys

REPO_URL = "https://github.com/Gulshan-heap/form-analyser.git"
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")

if not os.path.exists("train_dl.py"):                       # not already inside the repo
    os.chdir("/kaggle/working" if IN_KAGGLE else "/content" if IN_COLAB else ".")
    if not os.path.exists("form-analyser"):
        r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL], capture_output=True, text=True)
        if r.returncode != 0 and IN_KAGGLE:                 # Internet off: use the repo uploaded as a Kaggle Dataset
            found = glob.glob("/kaggle/input/**/train_dl.py", recursive=True)
            assert found, "Turn Internet ON in Session options, or add the repo as a Kaggle Dataset.\n" + r.stderr
            shutil.copytree(os.path.dirname(found[0]), "form-analyser")
        elif r.returncode != 0:
            raise RuntimeError(r.stderr)
    os.chdir("form-analyser")
sys.path.insert(0, os.getcwd())
print("working in", os.getcwd())

In [ ]:
# --- 2. Imports and device ---
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
import train_dl as T                                        # all model / metric code lives here

FORCE_CPU = False        # set True if a GPU error appears (e.g. an old Kaggle P100). CPU is fast enough for this model.
if FORCE_CPU:
    T.DEVICE = "cpu"
print("torch", torch.__version__, "| device:", T.DEVICE,
      "|", torch.cuda.get_device_name(0) if T.DEVICE == "cuda" else "no GPU (that is fine)")

## 2. The data
One row per rep. `invalid` = the judge said *no-rep* (hips not below the knees, or no full extension).

In [ ]:
d, x, dur, y = T.load("datasets/sequences_cfrep.npz")
person, view, yn = d["person"], d["view"], d["y"]
print("sequence tensor:", tuple(x.shape), "= (reps, channels, time steps); channels:", d["channels"].tolist())
df = pd.DataFrame(dict(person=person, view=view, invalid=yn))
label = df.invalid.map({0: "valid", 1: "invalid"})
print(pd.crosstab(df.person, label, margins=True), "\n")
print(pd.crosstab(df.view, label, margins=True))

In [ ]:
# What the network sees: mean knee-angle curve of valid vs invalid reps
t = np.linspace(0, 100, x.shape[-1])
for lab, name, c in ((0, "valid", "tab:green"), (1, "invalid", "tab:red")):
    k = d["x"][yn == lab, 0] * 180
    plt.plot(t, k.mean(0), c=c, label=f"{name} (n={len(k)})")
    plt.fill_between(t, k.mean(0) - k.std(0), k.mean(0) + k.std(0), color=c, alpha=.15)
plt.xlabel("% of rep"); plt.ylabel("knee angle (deg)"); plt.title("Knee angle through a rep"); plt.legend(); plt.show()

## 3. Settings
More seeds and epochs = steadier numbers but slower. The defaults take a few minutes on CPU.

In [ ]:
ARCHS  = ["cnn", "lstm"]     # "cnn" = 1D convolutions, "lstm" = bidirectional LSTM
EPOCHS = 80
SEEDS  = 3                   # models averaged per held-out person (small data is noisy)

## 4. Leave-one-person-out evaluation
Rule baselines first (a single threshold learned on the training people), then the deep models.

In [ ]:
results = {}                                                   # name -> (predicted invalid?, score)
for name, key, low in (("Rule: hip below knee", "max_hbk", True), ("Rule: knee angle", "min_knee", False)):
    results[name] = T.rule_baseline(d, key, low)

for arch in ARCHS:
    print("training", arch.upper(), "...", end=" ")
    prob = T.lopo_probs(arch, x, dur, y, person, EPOCHS, SEEDS)
    results[f"DL: {arch.upper()}"] = (prob >= 0.5, prob)
    print("done")

In [ ]:
rows = []
for name, (pred, score) in results.items():
    m = T.metrics(yn, pred, score)
    row = {"method": name, **{k: round(m[k], 3) for k in ("precision", "recall", "f1", "balanced_acc", "auc")}}
    for v in ("front", "diagonal", "side"):
        row[f"F1 {v}"] = round(T.f1(yn[view == v], pred[view == v]), 3)
    rows.append(row)
table = pd.DataFrame(rows).set_index("method")
os.makedirs("results", exist_ok=True)
table.to_csv("results/dl_results.csv")
table

In [ ]:
fig, ax = plt.subplots(1, len(results), figsize=(3.2 * len(results), 3.2))
for a, (name, (pred, score)) in zip(np.atleast_1d(ax), results.items()):
    m = T.metrics(yn, pred, score)
    a.imshow([[m["tn"], m["fp"]], [m["fn"], m["tp"]]], cmap="Blues")
    for i, j, v in ((0, 0, m["tn"]), (0, 1, m["fp"]), (1, 0, m["fn"]), (1, 1, m["tp"])):
        a.text(j, i, v, ha="center", va="center", fontsize=13)
    a.set_xticks([0, 1], ["pred valid", "pred invalid"]); a.set_yticks([0, 1], ["valid", "invalid"])
    a.set_title(name, fontsize=9)
plt.tight_layout(); plt.show()

table[["F1 front", "F1 diagonal", "F1 side"]].T.plot.bar(figsize=(7, 3), rot=0,
                                                          title="F1 for invalid reps by camera view (viewpoint study)")
plt.ylabel("F1"); plt.show()

## 5. Where does it go wrong?
Look at the reps the best deep model gets wrong - these are your failure cases for the report
(occlusion, a person standing very deep, bad pose tracking, a borderline judge call...).

In [ ]:
best = max((n for n in results if n.startswith("DL")), key=lambda n: T.f1(yn, results[n][0]))
pred, prob = results[best]
err = pd.DataFrame(dict(clip=d["clip"], rep=d["idx"], person=person, view=view, truth=np.where(yn, "invalid", "valid"),
                        P_invalid=prob.round(2), pose_found=d["found"].round(2)))[pred != (yn == 1)]
print(best, "-", len(err), "wrong reps out of", len(yn))
err.sort_values("P_invalid").head(40)

## 6. Save the final model and results
Trains one model on **all** people. Commit `results/` and `models/` back to the repo (or send them to the team).

In [ ]:
FINAL_ARCH = best.split(": ")[1].lower()
model = T.train_one(FINAL_ARCH, x, dur, y, EPOCHS, seed=0)
os.makedirs("models", exist_ok=True)
path = f"models/analyser_b_{FINAL_ARCH}.pt"
torch.save(dict(arch=FINAL_ARCH, state=model.state_dict(), channels=d["channels"].tolist(), seq_len=x.shape[-1]), path)
pd.DataFrame(dict(clip=d["clip"], rep=d["idx"], person=person, view=view, truth=yn,
                  **{n: s.round(4) for n, (p, s) in results.items()})).to_csv("results/dl_predictions.csv", index=False)
print("saved", path, "and results/dl_results.csv, results/dl_predictions.csv")

if IN_COLAB:                                                # Colab: download to your laptop
    from google.colab import files
    for f in (path, "results/dl_results.csv", "results/dl_predictions.csv"):
        files.download(f)
elif IN_KAGGLE:
    print("Kaggle: find the files under 'Output' (right panel) after the notebook finishes - use 'Save Version'.")